# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the Mass Spectrometry dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset metadata (Croissant schema) is provided via the following URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

We will use the unique `@id` for all dataset entities (record sets, fields, columns) as recommended by FAIR practices.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and instantiate the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (not as a dict)
metadata = dataset.metadata
print(metadata)

## 2. Data Overview
List all available record sets, their `@id`s, and the contained fields/columns. This overview guides data extraction and processing.

---
**Tip:** All entities are referenced by their `@id`. 
---

In [ ]:
# List all available record sets and their fields
print("Available record sets (with @id and label):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  @id: {rs.id}")
    print(f"     Name: {rs.name}")
    if rs.fields:
        print("     Fields (with @id and label):")
        for f in rs.fields:
            print(f"       {f.id} ({f.name})")
    print("")

Let's also see a sample record from each record set.

In [ ]:
for rs in record_sets:
    print(f"First record in record set: {rs.id} ({rs.name})")
    try:
        record_gen = dataset.records(record_set=rs.id)
        record = next(record_gen)
        print(json.dumps(record, indent=2)[:500])
    except Exception as e:
        print(f"  Could not load: {e}")
    print("-")

## 3. Data Extraction
Select record sets for analysis using their `@id`s, and load the tabular content into Pandas DataFrames for exploration.

*Pick record sets you want to analyze. Here we extract all available record sets.*

In [ ]:
# Load data for each record set into a DataFrame, indexed by its @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    print(f"Loading records from {rs_id} ...")
    try:
        recs = list(dataset.records(record_set=rs_id))
        if len(recs):
            df = pd.DataFrame(recs)
            dataframes[rs_id] = df
            print("  Columns:", df.columns.tolist())
            display(df.head(2))
        else:
            print("  No data records found.")
    except Exception as e:
        print(f"  Error: {e}")
    print("---")

Below, pick one of the main record sets (with substantial, tabular data) for further analysis. For this dataset, we'll typically have a record set with protein information; replace `main_record_set_id` with the actual one of interest as observed in the output above.

In [ ]:
# Define the main record set to analyze (edit as needed based on above output!)
# Example: main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None  # update this for your use-case

# Show DataFrame info
if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"DataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())
else:
    print(f"No DataFrame found for {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Now, let's process the data in the main record set. 
- Use the entity `@id`s to refer to fields/columns. 
- Pick a numeric column (e.g., protein intensity, molecular weight, etc.) and a group/categorical column (such as 'description' or 'sample type').

Replace the placeholder variable values as appropriate for this dataset.


In [ ]:
# Pick column names based on the DataFrame's output above:
# Example: numeric_field = 'Abundance', group_field = 'Description'
numeric_field = None
group_field = None
if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    potential_numeric_fields = df.select_dtypes(include='number').columns.tolist()
    print("Numeric fields detected:", potential_numeric_fields)
    if len(potential_numeric_fields) > 0:
        numeric_field = potential_numeric_fields[0]  # pick the first numeric field as example
    # For grouping, pick a non-numeric column
    for col in df.columns:
        if col not in potential_numeric_fields:
            group_field = col
            break

    print(f"Using numeric_field: {numeric_field} | group_field: {group_field}")
    if numeric_field is not None:
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        print(f"Filtering for values above the mean ({threshold}) in {numeric_field}")
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize selected numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by group_field
        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields available for analysis.")
else:
    print("Main record set not loaded into dataframe.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and the mean per group, if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Plot distribution
if main_record_set_id in dataframes and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    # If there is a group_field, visualize group means
    if group_field is not None and group_field in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(10,5))
        group_means = dataframes[main_record_set_id].groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        group_means.plot(kind='bar')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- The dataset was parsed using `mlcroissant` and explored by record sets and field `@id`s.
- We performed a basic exploratory analysis, filtered, normalized, and visualized a selected numeric field, and grouped results by a categorical field.
- You may extend this notebook by selecting different record sets or fields according to your research question.

**Next:** See the [mlcroissant documentation](https://mlcommons.github.io/croissant/) for more advanced integration and automation.